In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

2.21.0


In [ ]:
datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

In [ ]:
train_data = datagen.flow_from_directory(
    'dataset/',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    'dataset/',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
test_datagen = ImageDataGenerator(rescale=1.0/255)

test_data = test_datagen.flow_from_directory(
    'test/',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224,224,3))

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dropout(0.5)(x)

x = Dense(256, activation='relu')(x)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)

predictions = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=100,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.show()

In [ ]:
preds = model.predict(test_data)

y_pred = np.argmax(preds, axis=1)
y_true = test_data.classes

In [ ]:
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.show()

In [ ]:
print(classification_report(y_true, y_pred))

In [ ]:
fpr, tpr, _ = roc_curve(y_true, preds[:,1])
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label="AUC="+str(roc_auc))
plt.legend()
plt.show()

In [2]:
import os
print(os.listdir())

['.android', '.aws', '.azure', '.bash_history', '.config', '.cursor', '.docker', '.dotnet', '.emulator_console_auth_token', '.gitconfig', '.gradle', '.ipynb_checkpoints', '.ipython', '.jdks', '.jupyter', '.keras', '.lesshst', '.m2', '.matplotlib', '.ms-ad', '.node_repl_history', '.PreviewWindow', '.prismic', '.python_history', '.QtWebEngineProcess', '.sao', '.ssh', '.th-client', '.ts_node_repl_history', '.viminfo', '.VirtualBox', '.vscode', '3dworld', 'AppData', 'Application Data', 'backend.zip', 'Contacts', 'Cookies', 'Creative Cloud Files', 'Creative Cloud Files leadymore5@gmail.com ce00e6862e150f133185a39ddf80248e1876abaf9951aa643d1672769547cb10', 'descktop-tutorial', 'desktop', 'docker_data', 'Documents', 'Downloads', 'edb_pgjdbc.exe', 'example', 'Favorites', 'frontend-stad', 'get-pip.py', 'Goruntu_Isleme_Ders_2', 'Hack', 'hackathons', 'holbertonschool-low_level_programming', 'IdeaProjects', 'IntelGraphicsProfiles', 'Java', 'Links', 'Local Settings', 'Machine_Learning', 'Main', 'ma

In [3]:
print(os.listdir('dataset'))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset'

In [4]:
import os
print(os.getcwd())

C:\Users\asus2


In [5]:
import os
os.chdir(r"C:\Users\asus2\OneDrive\Desktop\DeepLearningProject")
print(os.listdir())

['dataset', 'test']
